In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Hypothesis

Our general hypothesis is that the location of plants are disproportionately placed in marginalized communities. The characteristic of a marginalized community is not singular, which illicts the need to conduct multiple hypothesis testing to observe potential inequalities across multiple socioeconomic demographic metrics.

Our alternative hypotheses are as follows:
1) Plants are disproportionately located in areas with higher % People of Color, specifically: the difference in mean percentage of People of Color between counties with plants and counties without plants is X%. We chose this alternative hypothesis because we believe a X% difference in means would point to a clear discrepancy ????? I asked about how to choose this on ed still waiting for a response
2) Plants are disproportionately located in areas with higher % Low Income
3) Plants are disproportionately located in areas with higher % Less Than High School Education
4) Plants are disproportionately located in areas with higher % Unemployment Rate
5) There is a statistically significant difference in the % of Age under 5 for counties with plants vs without
6) There is a statistically significant difference in the % of Age over 64 for counties with plants vs without

Our null hypotheses are that there is no difference in means of these demographic factors between counties with plants and counties without plants.

We will be conducting A/B testing against each hypothesis. This is because we can treat each plant in the eGRID data as the 'treatment' of having a plant. The other counties across the US will be the control groups, for not having a plant. 

To correct for the multiple hypothesis tests, we will use two different methods:
- To control the FDR at 0.05, we will use the Benjamini–Yekutieli procedure, which controls the false discovery rate under arbitrary dependence assumptions. This is needed because the demographic metrics are not independent (source).
- To control for the FWER at 0.05, we will use the Bonferroni correction.


In [3]:
counties2 = pd.read_csv('data/mh_analysis_ready.csv', index_col=0)
counties2

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),Limited Life Expectancy (%),Plant primary fuel category_x,Plant annual net generation (MWh)
2,1001,AL,Autauga County,4,0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,NaN,NaN,NaN
3,1003,AL,Baldwin County,4,0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,NaN,NaN,NaN
4,1005,AL,Barbour County,4,0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,NaN,NaN,NaN
5,1007,AL,Bibb County,4,0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,NaN,NaN,NaN
6,1009,AL,Blount County,4,0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7805,51097,VA,King and Queen,3,1,287.0,28.000000,26.000000,8.000000,0.000000,1.000000,26.000000,4.000000,18.0,BIOMASS,"53,765"
7181,48347,TX,Nacogdoches,6,1,424.0,15.000000,40.000000,10.000000,0.000000,3.000000,20.000000,4.000000,4.0,BIOMASS,"249,859"
3670,16009,ID,Benewah,10,1,1477.0,37.000000,39.000000,13.000000,1.000000,4.000000,18.000000,5.000000,23.0,BIOMASS,"8,800"
192,1069,AL,Houston,4,1,107.0,29.000000,47.000000,21.000000,0.000000,12.000000,25.000000,4.000000,25.0,NaN,"14,758,529"


In [4]:
# 1 row per county - this was overwriting how egrid calculated the demographic info, which i think is a more valuable method
#counties2 =  pd.read_csv('data/mh_analysis_2.csv', index_col=0)
#counties2

In [15]:
numerical_col = 'Under Age 5 (%)' #, 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)' , 'Over Age 64 (%)', 'Under Age 5 (%)']
binary_col = 'has_plant'

shuffled_table = counties2.copy()
    
shuffled_labels = counties2.sample(replace=False, frac = 1, random_state = 1)[binary_col]
shuf = shuffled_labels.to_numpy()
shuffled_table['Shuffled Label'] = shuf
selected_shuf = shuffled_table.loc[:, (numerical_col, 'Shuffled Label')]

series = selected_shuf.groupby('Shuffled Label').mean().loc[:, numerical_col]
dif = series.iloc[1] - series.iloc[0]
#hypothesis: series.iloc[1] > series.iloc[0] ie dif >0
selected = counties2.loc[:, (numerical_col, binary_col)]
series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col] #iloc[1] is has_plant == 1, iloc[0] is has_plant == 0
observed_difference = series_obs.iloc[1] - series_obs.iloc[0] # one sided alternative hypothesis
observed_difference

-0.3635584061333432

In [16]:
series

Shuffled Label
0    5.578420
1    5.565388
Name: Under Age 5 (%), dtype: float64

In [6]:
shuffled_table


,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),Limited Life Expectancy (%),Plant primary fuel category_x,Plant annual net generation (MWh),Shuffled Label
2,1001,AL,Autauga County,4,0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,NaN,NaN,NaN,0
3,1003,AL,Baldwin County,4,0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,NaN,NaN,NaN,0
4,1005,AL,Barbour County,4,0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,NaN,NaN,NaN,0
5,1007,AL,Bibb County,4,0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,NaN,NaN,NaN,0
6,1009,AL,Blount County,4,0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7805,51097,VA,King and Queen,3,1,287.0,28.000000,26.000000,8.000000,0.000000,1.000000,26.000000,4.000000,18.0,BIOMASS,"53,765",0
7181,48347,TX,Nacogdoches,6,1,424.0,15.000000,40.000000,10.000000,0.000000,3.000000,20.000000,4.000000,4.0,BIOMASS,"249,859",0
3670,16009,ID,Benewah,10,1,1477.0,37.000000,39.000000,13.000000,1.000000,4.000000,18.000000,5.000000,23.0,BIOMASS,"8,800",0
192,1069,AL,Houston,4,1,107.0,29.000000,47.000000,21.000000,0.000000,12.000000,25.000000,4.000000,25.0,NaN,"14,758,529",1


In [23]:
def difference_of_means(table, group_label, numerical_col, abs_dif):
    
    series = table.groupby('Shuffled Label').mean().loc[:, numerical_col]
    if abs_dif == True:
        return abs(series.iloc[1] - series.iloc[0])
    else:
        return series.iloc[1] - series.iloc[0]

In [24]:
def one_simulated_difference_of_means(df, numerical_col, binary_col, i, abs_dif):

    shuffled_table = df.copy()
    
    shuffled_labels = df.sample(replace=False, frac = 1, random_state = i)[binary_col]
    shuf = shuffled_labels.to_numpy()
    shuffled_table['Shuffled Label'] = shuf
    selected_shuf = shuffled_table.loc[:, (numerical_col, 'Shuffled Label')]
    
    return difference_of_means(selected_shuf, 'Shuffled Label', numerical_col, abs_dif)   

In [25]:
def avg_difference_in_means(df, numerical_col, abs_dif, binary_col= 'has_plant'):
    """
   The function computes the p-value for a test of the following hypothesis test:
        H0 : There is no difference in the average value of numerical_col between the two
            groups specified in binary_col.
        H1 : The average value of numerical_col is different for the two groups specified
            in binary_col
    inputs
        numerical_col: a numerical column name
        binary_col: a binary column name
    """
    selected = df.loc[:, (numerical_col, binary_col)]
    series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col]
    if abs_dif == True:
        observed_difference = abs(series_obs.iloc[1] - series_obs.iloc[0])
    else: 
        observed_difference = series_obs.iloc[1] - series_obs.iloc[0]
    differences = []

    repetitions = 500
    for i in np.arange(repetitions):
        new_difference = one_simulated_difference_of_means(df, numerical_col, binary_col, i, abs_dif)
        differences = np.append(differences, new_difference)                               

    empirical_p = np.count_nonzero(differences >= observed_difference) / repetitions #how many samples have as extreme of a difference?

    #print(f' observed dif: {observed_difference}, empirical p: {empirical_p}')
    return empirical_p
    

In [26]:
one_sided_cols = ['People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)']
two_sided_cols = ['Over Age 64 (%)', 'Under Age 5 (%)']
binary_col = ['has_plant']
pvals = {}

for i in one_sided_cols:
    pvals[f'{i} and {binary_col}'] = avg_difference_in_means(counties2, i, abs_dif = True)
for k in two_sided_cols:
    pvals[f'{k} and {binary_col}'] = avg_difference_in_means(counties2, k, abs_dif = False)

pvals

{"People of Color (%) and ['has_plant']": 0.0,
 "Low Income (%) and ['has_plant']": 0.0,
 "Less Than High School Education (%) and ['has_plant']": 0.012,
 "Unemployment Rate (%) and ['has_plant']": 0.404,
 "Over Age 64 (%) and ['has_plant']": 0.016,
 "Under Age 5 (%) and ['has_plant']": 1.0}

In [29]:
# analysis by regions

for r in range(1, 10):
    print(f'Starting analysis for EPA Region {r}')
    one_sided_cols = ['People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)']
    two_sided_cols = ['Over Age 64 (%)', 'Under Age 5 (%)']
    binary_col = ['has_plant']
    pvals = {}

    regional_df = counties2[counties2['EPA Region']== r]
    for i in one_sided_cols:
        pvals[f'{i} and {binary_col}'] = avg_difference_in_means(regional_df, i, abs_dif = True)
    for k in two_sided_cols:
        pvals[f'{k} and {binary_col}'] = avg_difference_in_means(regional_df, k, abs_dif = False)
    
    display(pvals)

Starting analysis for EPA Region 1


{"People of Color (%) and ['has_plant']": 0.21,
 "Low Income (%) and ['has_plant']": 0.394,
 "Less Than High School Education (%) and ['has_plant']": 0.134,
 "Unemployment Rate (%) and ['has_plant']": 0.024,
 "Over Age 64 (%) and ['has_plant']": 0.866,
 "Under Age 5 (%) and ['has_plant']": 0.982}

Starting analysis for EPA Region 2


{"People of Color (%) and ['has_plant']": 0.882,
 "Low Income (%) and ['has_plant']": 0.62,
 "Less Than High School Education (%) and ['has_plant']": 0.934,
 "Unemployment Rate (%) and ['has_plant']": 0.682,
 "Over Age 64 (%) and ['has_plant']": 0.1,
 "Under Age 5 (%) and ['has_plant']": 0.844}

Starting analysis for EPA Region 3


{"People of Color (%) and ['has_plant']": 0.804,
 "Low Income (%) and ['has_plant']": 0.33,
 "Less Than High School Education (%) and ['has_plant']": 0.12,
 "Unemployment Rate (%) and ['has_plant']": 0.372,
 "Over Age 64 (%) and ['has_plant']": 0.162,
 "Under Age 5 (%) and ['has_plant']": 0.788}

Starting analysis for EPA Region 4


KeyboardInterrupt: 

In [10]:
fwer = 0.05
num_tests = len(numerical_cols) * len(binary_cols)
fwer_threshold = fwer / num_tests
print(f'FWER threshold: {fwer_threshold}')
reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fwer_threshold]
#list(pvals.values())
reject_null

FWER threshold: 0.008333333333333333


["People of Color (%) and ['has_plant']"]

In [11]:


p_sorted = sorted(list(pvals.values()))

m = len(p_sorted)  
k = np.arange(1, m+1)  # index of each test in sorted order
alpha = 0.05
c_m = np.sum([1/i for i in range(1, m)])
compare = (alpha * k) /(m* c_m)
below_than = p_sorted <= compare
cols = {'k' : k, 'p-vals' : p_sorted, 'compare': compare, 'below than' : below_than}

ps = pd.DataFrame(cols)
ps


,k,p-vals,compare,below than
0,1,0.0000,0.003650,True
1,2,0.0124,0.007299,False
2,3,0.7828,0.010949,False
3,4,0.9952,0.014599,False
4,5,0.9988,0.018248,False
5,6,1.0000,0.021898,False


In [12]:
fdr_threshold = ps[ps['below than'] == True]['p-vals'].iloc[-1]
print(f'FDR threshold: {fdr_threshold}')
fdr_reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fdr_threshold]
#list(pvals.values())
fdr_reject_null

FDR threshold: 0.0


["People of Color (%) and ['has_plant']"]

### Results
- Summarize and interpret the results from the hypothesis tests themselves.
- 